# Understanding Data





In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Datasets:

In [2]:
orders = pd.read_csv("../Data/Raw/olist_orders_dataset.csv")
order_items = pd.read_csv("../Data/Raw/olist_order_items_dataset.csv")
customers = pd.read_csv("../Data/Raw/olist_customers_dataset.csv")
payments = pd.read_csv("../Data/Raw/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../Data/Raw/olist_order_reviews_dataset.csv")
sellers =pd.read_csv("../Data/Raw/olist_sellers_dataset.csv")
products = pd.read_csv("../Data/Raw/olist_products_dataset.csv")
product_name_translation = pd.read_csv("../Data/Raw/product_category_name_translation.csv")
geolocation = pd.read_csv("../Data/Raw/olist_geolocation_dataset.csv")


## Datasets inspection:

### insepction function

In [3]:
def inspect_dataframe(df):

    print("--- Number of Rows and Columns ---")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

    print("\n--- Data Type ---")
    print(df.dtypes)

    print("\n--- Duplicates ---")
    print(f"duplicate rows: {df.duplicated().sum()}")

    print("\n--- Missing Values ---")
    missing = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": df.isna().mean() * 100
    })

    print(missing[missing["missing_count"] > 0])


### Orders Table:

**purpose:** it stores general information about customer's order including order status and purchasing,approval,shipping,delivery timestamps

**Tip : each row represent one order**

**important columns:**
- **order_id:** Unique identifier of an order
- **customer_id:** Identifier of customers associated with the order
- **order_status:** current or recorded status of the order (created, approved, invoiced, canceled, unavailable, processing, delivered)
- **timestamps columns:** time & date of each satus

**relational columns:**
- order_id = primary key(Unique)
- customer_id = foreign key


In [4]:
inspect_dataframe(orders)

--- Number of Rows and Columns ---
Rows: 99441
Columns: 8

--- Data Type ---
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

--- Duplicates ---
duplicate rows: 0

--- Missing Values ---
                               missing_count  missing_percent
order_approved_at                        160         0.160899
order_delivered_carrier_date            1783         1.793023
order_delivered_customer_date           2965         2.981668


### order_items Table:

**purpose:** it stores information about order items including product,seller,price,shipping cost and deadline

**Tip : each row represent an order item**

**important columns:**
- **order_id:**  identifier of an order
- **order_item_id:** Specifies the item number within the order.
- **product_id:** Identifier of the purchased product
- **seller_id:** Identifier of the seller
- **shipping_limit_date:** Deadline for the seller to ship the item
- **price:** Price of the item
- **freight_value:** Shipping cost associated with the item


**relational columns:**
- order_id = foreign key
- product_id = foreign key
- seller_id = foreign key

**tip : it has no unique column**

In [5]:
inspect_dataframe(order_items)

--- Number of Rows and Columns ---
Rows: 112650
Columns: 7

--- Data Type ---
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

--- Duplicates ---
duplicate rows: 0

--- Missing Values ---
Empty DataFrame
Columns: [missing_count, missing_percent]
Index: []


### order_payments Table:

**purpose:** it Stores payment information related with customer orders, including payment methods, installment details, and payment amounts

**Tip : Each row represents a payment record associated with an order(An order may have multiple payment records)**

**important columns:**
- **order_id**: Identifier of the associated order; links this table to the orders table
- **payment_sequential:** Sequential number identifying a payment record within an order
- **payment_type:** Payment method, such as credit card or boleto or voucher
- **payment_installments:** Number of installments used for the payment
- **payment_value:** value of the payment

**relational columns:**
- order_id = foreign key

**tip : it has no unique column**

In [6]:
inspect_dataframe(payments)

--- Number of Rows and Columns ---
Rows: 103886
Columns: 5

--- Data Type ---
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

--- Duplicates ---
duplicate rows: 0

--- Missing Values ---
Empty DataFrame
Columns: [missing_count, missing_percent]
Index: []


### order_reviews Table:

**purpose:** it Stores customer reviews and ratings related with orders, including review comments and timestamps

**Tip : Each row represents a review record associated with an order**

**important columns:**
- **review_id:** Identifier of the review
- **order_id:** Identifier of the associated order 
- **review_score:** Customer's rating of the order(in range of 1 to 5)
- **review_comment_title:** Title of the review
- **review_comment_message:** Text of the customer's review
- **review_creation_date:** Date the satisfaction survey was sent to the customer (not the date the customer answered it)
- **review_answer_timestamp:** Timestamp of when the customer answered the survey

**relational columns:**
- order_id = foreign key

**tip : it has no unique column**

In [7]:
inspect_dataframe(reviews)

--- Number of Rows and Columns ---
Rows: 99224
Columns: 7

--- Data Type ---
review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str
dtype: object

--- Duplicates ---
duplicate rows: 0

--- Missing Values ---
                        missing_count  missing_percent
review_comment_title            87656        88.341530
review_comment_message          58247        58.702532


### customers Table:

**purpose:** it stores information about customer including customer identifiers and geographic location

**Tip : each row represent a customer with customer_id**

**important columns:**
- **customer_id:** Identifier used to associate a customer record with orders
- **customer_unique_id:** Identifier intended to identify a customer across multiple customer records
- **customer_zip_code_prefix:** Prefix of the customer's postal code
- **customer_city:** City associated with the customer
- **customer_state:** Brazilian state associated with the customer

**relational columns:**
- customer_id = primary key(Unique)
- customer_zip_code_prefix = join key to geolocation (not a true foreign key: geolocation has many rows per prefix and some prefixes are missing, so it must be aggregated to one row per prefix before joining)

**Customer Identifiers: `customer_id` vs. `customer_unique_id`**

- `customer_id`: Identifies a customer record in the dataset and is used to link orders to customer records.
- `customer_unique_id`: Identifies a unique customer across multiple customer records.

A single customer may have multiple `customer_id` values while sharing the same `customer_unique_id`.

**Analytical result:**

- Use `customer_id` to identify customer records and link them to orders.
- Use `customer_unique_id` to estimate the number of unique customers.

Therefore, the number of unique `customer_id` values may differ from the number of unique `customer_unique_id` values.

In [8]:
inspect_dataframe(customers)

--- Number of Rows and Columns ---
Rows: 99441
Columns: 5

--- Data Type ---
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

--- Duplicates ---
duplicate rows: 0

--- Missing Values ---
Empty DataFrame
Columns: [missing_count, missing_percent]
Index: []


### seller Table:

**purpose:** it stores information about seller including seller identifiers and geographic location

**Tip : each row represent a seller with seller_id**

**important columns:**
- **seller_id:** Unique identifier of seller
- **seller_zip_code_prefix:** Prefix of the seller's postal code
- **seller_city:** City associated with the seller
- **seller_state:** Brazilian state associated with the seller

**relational columns:**
- seller_id = primary key(Unique)
- seller_zip_code_prefix = join key to geolocation (aggregate geolocation to one row per prefix before joining)

In [9]:
inspect_dataframe(sellers)

--- Number of Rows and Columns ---
Rows: 3095
Columns: 4

--- Data Type ---
seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object

--- Duplicates ---
duplicate rows: 0

--- Missing Values ---
Empty DataFrame
Columns: [missing_count, missing_percent]
Index: []


### products Table:

**purpose:** it Stores product-related information, including product categories, name and description information, and physical dimensions

**Tip : Each row represents a product**

**important columns:**
- **product_id:** Unique identifier of a product
- **product_category_name:** Category to which the product belongs
- **product_name_length:** Length of the product name, typically measured in characters
- **product_description_length:** Length of the product description, typically measured in characters
- **product_photos_qty:** Number of photos associated with the product
- **product_weight/length/hieght/width_g/cm:** Product weight and dimensions in grams and centimeters

**relational columns:**
- product_id = primary key(Unique)


In [10]:
inspect_dataframe(products)

--- Number of Rows and Columns ---
Rows: 32951
Columns: 9

--- Data Type ---
product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

--- Duplicates ---
duplicate rows: 0

--- Missing Values ---
                            missing_count  missing_percent
product_category_name                 610         1.851234
product_name_lenght                   610         1.851234
product_description_lenght            610         1.851234
product_photos_qty                    610         1.851234
product_weight_g                        2         0.006070
product_length_cm                       2         0.006070
product_height_cm                       2         0.006070
product_width_cm                 

### geolocation Table:

**purpose:** it Stores geographic information associated with Brazilian postal code prefixes, including latitude, longitude, city, and state

**Tip : Each row represents a geolocation record associated with a postal code prefix**

**important columns:**
- **geolocation_zip_code_prefix:** Brazilian postal code prefix associated with the geographic record
- **geolocation_lat:** Latitude coordinate of the geographic location
- **geolocation_lng:** Longitude coordinate of the geographic location
- **geolocation_city:** City associated with the location
- **geolocation_state:** Brazilian state associated with the location

**relational columns:**
- geolocation_zip_code_prefix = join key for customers and sellers (not unique: one prefix has many coordinate rows)

In [11]:
inspect_dataframe(geolocation)

--- Number of Rows and Columns ---
Rows: 1000163
Columns: 5

--- Data Type ---
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

--- Duplicates ---
duplicate rows: 261831

--- Missing Values ---
Empty DataFrame
Columns: [missing_count, missing_percent]
Index: []


## General Report :

### Dataset Structure and Relationships :

The Olist dataset consists of multiple interconnected tables representing different aspects of the e-commerce business, including customers, orders, order items, products, sellers, payments, reviews, and geolocation data.

The `orders` table serves as the central table, connecting each order to a customer record through `customer_id`. The `customers` table then provides `customer_unique_id`, which identifies the same real customer across several orders. The `order_items` table links orders to products and sellers, while the `order_payments` and `order_reviews` tables provide additional information about payment transactions and customer feedback. The `products` and `sellers` tables contain product and seller details, respectively, and the `geolocation` table provides geographic information based on ZIP code prefixes.

Together, these relationships provide a foundation for analyzing sales performance, customer behavior, product performance, delivery processes, and customer satisfaction.